# 🎙️ Pipeline Fallback Step: Audio Splitting: SoundFile Block Reading

Performs memory-efficient audio splitting by reading specific frames directly from disk using SoundFile.

## Environment Setup

In [ ]:
import soundfile as sf
import json
import os
import numpy as np

## Google Drive Mount & Form Configuration

In [ ]:
try:
    drive.mount('/content/drive')
    print("Google Drive successfully mounted.")
except Exception as e:
    print(f"Drive mount error: {e}")

# @markdown ### 📂 Splitting Configuration
input_audio_path = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/standardized.wav" # @param {type:"string"}
timeline_json_path = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/maraulikhurad3_refined_timeline.json" # @param {type:"string"}
output_dir = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/isolated_speakers/" # @param {type:"string"}

os.makedirs(output_dir, exist_ok=True)

## Execute Fallback Processing

In [ ]:
try:
    with open(timeline_json_path, "r", encoding="utf-8") as f:
        segments = json.load(f)
        
    def parse_time(time_str, idx):
        part = time_str.split('-')[idx].replace('[','').replace(']','').strip()
        parts = part.split(':')
        return float(parts[-2])*60 + float(parts[-1])

    os.makedirs(output_dir, exist_ok=True)
    
    with sf.SoundFile(input_audio_path) as audio:
        sr = audio.samplerate
        speaker_samples = {}
        
        for entry in segments:
            start_sec = parse_time(entry['time'], 0)
            end_sec = parse_time(entry['time'], 1)
            speaker = entry['speaker']
            
            start_frame = int(start_sec * sr)
            num_frames = int((end_sec - start_sec) * sr)
            
            audio.seek(start_frame)
            chunk = audio.read(num_frames)
            
            if speaker not in speaker_samples:
                speaker_samples[speaker] = [chunk]
            else:
                speaker_samples[speaker].append(chunk)
                
        for speaker, samples_list in speaker_samples.items():
            concatenated = np.concatenate(samples_list, axis=0)
            out_file = os.path.join(output_dir, f"{speaker}_isolated.wav")
            sf.write(out_file, concatenated, sr, format='WAV', subtype='PCM_16')
            print(f"Exported speaker isolated file: {out_file}")
            
    print("\n[SUCCESS] SoundFile block splitting complete!")
except Exception as e:
    print(f"\n[ERROR] SoundFile block splitting failed: {e}")